# Limpeza de `usuarios.csv`

**Objetivo:** gerar o cadastro de contas pronto para análise em `dados/processados/usuarios_limpa.parquet`.

**Regras aplicadas** (todas registradas no `decisoes.md`; o diagnóstico está no notebook 07):

| Passo | Regra | Origem |
|---|---|---|
| 1 | Tirar as contas de teste (e-mail `@vitaliza.com.br`) | decisão de 31/10/2026 |
| 2 | Tirar as 959 contas-cópia (e-mail em maiúsculas, a conta mais nova do par), incluindo as 119 com cadastro depois do corte | decisão de 07/11/2026 |
| 3 | `data_cadastro` só como data, em horário de Brasília | decisão de 07/11/2026 |
| 4 | Nascimento `1900-01-01`, `2031-05-12` e vazio viram ausente; criar `idade_no_corte` | decisão de 07/11/2026 |
| 5 | Padronizar `sexo`, `sistema`, `cidade` e `cep_prefixo` | decisões de 07/11/2026 |
| 6 | Marcar `cadastro_antes_do_log` (cadastro antes de 01/07/2024) | decisão de 07/11/2026 |

**O e-mail não entra na tabela limpa:** depois dos passos 1 e 2 nenhuma análise precisa dele (aprovado em 07/11/2026).

**Atenção ao usar:** `versao_app`, `relogio_conectado` e `sistema` mostram a conta no dia da exportação, sem data.
Servem para descrever a base, mas não como preditor de saída (decisão de 07/11/2026). Contas com
`cadastro_antes_do_log` não servem para taxa de conversão nem funil grátis → pago.

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.
As linhas antes e depois de cada passo vão para `saidas/limpeza_usuarios_registro.csv`.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
INICIO_LOG = pd.Timestamp("2024-07-01")   # primeiro mês de eventos_assinatura.csv e pagamentos.csv
CORTE = pd.Timestamp("2026-09-30")

ARQUIVOS = ["usuarios.csv", "onboarding.csv", "pagamentos.csv", "atividades.parquet", "amizades.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas)")
    return df

## 0. Leitura

Tudo como texto (`dtype=str, keep_default_na=False`): o arquivo chega como foi gravado e vazio fica `""`.

In [2]:
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False)
assert len(usuarios) == 64_948   # cabeçalho da seção 02 do dicionário
assert usuarios["user_id"].is_unique

base = registrar("0. Arquivo bruto", "usuarios.csv sem filtro", usuarios.copy())

0. Arquivo bruto: 64,948 linhas (0 removidas)


## 1. Contas de teste

As 60 contas com e-mail `@vitaliza.com.br` são exatamente as 60 do canal `interno`.

In [3]:
email_norm = base["email"].str.strip().str.lower()
eh_teste = email_norm.str.endswith("@vitaliza.com.br")
assert eh_teste.sum() == 60
assert (eh_teste == (base["canal_aquisicao"] == "interno")).all()

base = registrar("1. Sem contas de teste", "e-mail não termina em @vitaliza.com.br", base.loc[~eh_teste].copy())

1. Sem contas de teste: 64,888 linhas (60 removidas)


## 2. Contas-cópia

Em 959 e-mails há duas contas. A cópia é a que tem o e-mail com letra maiúscula: sempre a mais nova do par, com o
perfil igual ao da original e sem nenhuma linha em outro arquivo do data room (notebook 07). As 119 contas com
cadastro depois do corte são todas cópias.

In [4]:
email = base["email"].str.strip()
base["_email_norm"] = email.str.lower()
base["_cadastro_hora"] = pd.to_datetime(base["data_cadastro"], format="%Y-%m-%d %H:%M:%S")
eh_copia = email != base["_email_norm"]

repetido = base["_email_norm"].duplicated(keep=False)
assert eh_copia.sum() == 959
assert (eh_copia <= repetido).all()                                     # toda cópia tem uma original
assert base.loc[repetido].groupby("_email_norm").size().eq(2).all()    # pares, nunca trios
mais_nova = base.loc[repetido].groupby("_email_norm")["_cadastro_hora"].transform("max") == base.loc[repetido, "_cadastro_hora"]
assert (mais_nova == eh_copia[repetido]).all()                          # a cópia é a mais nova do par
pos_corte = base["_cadastro_hora"].dt.normalize() > CORTE
print(f"cópias: {eh_copia.sum()}; delas, com cadastro depois do corte: {(eh_copia & pos_corte).sum()} de {pos_corte.sum()}")
assert (pos_corte <= eh_copia).all()

base = registrar("2. Sem contas-cópia", "e-mail com maiúscula (a conta mais nova de um e-mail repetido)",
                 base.loc[~eh_copia].copy())
assert base["_email_norm"].is_unique

cópias: 959; delas, com cadastro depois do corte: 119 de 119
2. Sem contas-cópia: 63,929 linhas (959 removidas)


## 3. Data de cadastro

O cadastro está em horário de Brasília (o teste grátis cai no mesmo dia em 100% das contas), mas a hora não é confiável:
antes de 07/2024 todas as contas estão às 12:00:00. Fica só a data.

In [5]:
base["data_cadastro"] = base["_cadastro_hora"].dt.normalize()
print(f"data_cadastro: {base['data_cadastro'].min().date()} a {base['data_cadastro'].max().date()}")
assert base["data_cadastro"].notna().all()
assert base["data_cadastro"].max() <= CORTE

base = registrar("3. Data de cadastro", "só a data, em horário de Brasília", base)

data_cadastro: 2020-09-01 a 2026-09-30
3. Data de cadastro: 63,929 linhas (0 removidas)


## 4. Nascimento e idade

`1900-01-01` é valor-padrão de formulário e `2031-05-12` é data no futuro: viram ausente, junto com os vazios.
`idade_no_corte` é a idade em anos completos em 30/09/2026.

In [6]:
PLACEHOLDERS = {"1900-01-01", "2031-05-12"}
print("trocados por ausente:", base["data_nascimento"].isin(PLACEHOLDERS).sum(), "| já vazios:", (base["data_nascimento"] == "").sum())
nasc = base["data_nascimento"].where(~base["data_nascimento"].isin(PLACEHOLDERS | {""}))
base["data_nascimento"] = pd.to_datetime(nasc, format="%Y-%m-%d")
fez_aniversario = (base["data_nascimento"].dt.month < CORTE.month) | (
    (base["data_nascimento"].dt.month == CORTE.month) & (base["data_nascimento"].dt.day <= CORTE.day))
base["idade_no_corte"] = (CORTE.year - base["data_nascimento"].dt.year - (~fez_aniversario).astype(int)).astype("Int64")
base.loc[base["data_nascimento"].isna(), "idade_no_corte"] = pd.NA

print(f"sem nascimento: {base['data_nascimento'].isna().sum():,}; idade de {base['idade_no_corte'].min()} a {base['idade_no_corte'].max()} anos")
assert base["idade_no_corte"].dropna().between(13, 100).all()   # limites plausíveis do diagnóstico (notebook 07)
assert (base["data_nascimento"].dropna() < base.loc[base["data_nascimento"].notna(), "data_cadastro"]).all()

base = registrar("4. Nascimento e idade", "1900-01-01, 2031-05-12 e vazio viram ausente; idade_no_corte", base)

trocados por ausente: 595 | já vazios: 3239


sem nascimento: 3,834; idade de 17 a 74 anos
4. Nascimento e idade: 63,929 linhas (0 removidas)


## 5. Categorias

- `sexo`: F, Fem, feminino → F; M, Masc, masculino → M; vazio → ausente;
- `sistema`: minúsculas (android, ios);
- `cidade`: 36 grafias → 13 cidades, pela mesma tabela do notebook 07;
- `cep_prefixo`: zero à esquerda reposto (os 2 dígitos da capital paulista); vazio → ausente.

In [7]:
SEXO = {"F": "F", "Fem": "F", "feminino": "F", "M": "M", "Masc": "M", "masculino": "M", "": None}
CIDADE = {
    "São Paulo": ["São Paulo", "Sampa", "são paulo", "São Paulo - SP", "Sao Paulo", "S. Paulo", "SAO PAULO", "SP"],
    "Rio de Janeiro": ["Rio de Janeiro", "Rio de janeiro", "rio de janeiro", "RIO DE JANEIRO", "RJ", "Rio"],
    "Belo Horizonte": ["Belo Horizonte", "Belo horizonte", "B. Horizonte", "BELO HORIZONTE", "BH"],
    "Florianópolis": ["Florianópolis", "floripa", "Florianopolis", "Floripa"],
    "Brasília": ["Brasília", "Brasília - DF", "BSB", "brasilia", "Brasilia"],
    "Campinas": ["Campinas"], "Curitiba": ["Curitiba"], "Fortaleza": ["Fortaleza"], "Goiânia": ["Goiânia"],
    "Porto Alegre": ["Porto Alegre"], "Recife": ["Recife"], "Salvador": ["Salvador"], "Santos": ["Santos"],
}
MAPA_CIDADE = {grafia: nome for nome, grafias in CIDADE.items() for grafia in grafias}
FAIXA_CEP = {"SP": (10, 199), "RJ": (200, 289), "MG": (300, 399), "BA": (400, 489), "PE": (500, 569),
             "CE": (600, 639), "DF": (700, 736), "GO": (737, 767), "PR": (800, 879), "SC": (880, 899), "RS": (900, 999)}

assert set(base["sexo"]) <= set(SEXO), set(base["sexo"]) - set(SEXO)
assert set(base["cidade"]) <= set(MAPA_CIDADE), set(base["cidade"]) - set(MAPA_CIDADE)
mudou = {
    "sexo": (base["sexo"] != base["sexo"].map(SEXO).fillna("")).sum(),
    "sistema": (base["sistema"] != base["sistema"].str.strip().str.lower()).sum(),
    "cidade": (base["cidade"] != base["cidade"].map(MAPA_CIDADE)).sum(),
    "cep_prefixo": (base["cep_prefixo"].str.len() == 2).sum(),
}
print("linhas padronizadas por coluna:", {k: int(v) for k, v in mudou.items()})

base["sexo"] = base["sexo"].map(SEXO)
base["sistema"] = base["sistema"].str.strip().str.lower()
base["cidade"] = base["cidade"].map(MAPA_CIDADE)
base["cep_prefixo"] = base["cep_prefixo"].where(base["cep_prefixo"] != "").str.zfill(3)

CATEGORICAS = {
    "sexo": {"F", "M", None},
    "sistema": {"android", "ios"},
    "uf": set(FAIXA_CEP),
    "canal_aquisicao": {"organico", "influenciador", "meta_ads", "indicacao", "google_ads", "parceria_corporativa"},
    "relogio_conectado": {"nenhum", "garmin", "apple_watch"},
    "versao_app": {"4.9.8", "5.1.0", "5.3.0", "5.3.2", "5.4.0", "5.4.1"},
}
for col, permitidos in CATEGORICAS.items():
    valores = set(base[col].where(base[col].notna(), None))
    assert valores <= permitidos, (col, valores - permitidos)
assert base.groupby("cidade")["uf"].nunique().eq(1).all()

cep = pd.to_numeric(base["cep_prefixo"])
faixa = base["uf"].map(FAIXA_CEP)
dentro = [pd.isna(c) or a <= c <= b for c, (a, b) in zip(cep, faixa)]
assert all(dentro)
assert base["cep_prefixo"].dropna().str.fullmatch(r"\d{3}").all()

display(base["cidade"].value_counts().to_frame("contas"))
base = registrar("5. Categorias", "sexo, sistema, cidade e cep_prefixo padronizados", base)

linhas padronizadas por coluna: {'sexo': 24430, 'sistema': 15824, 'cidade': 13393, 'cep_prefixo': 1082}


,contas
cidade,
São Paulo,27035
Rio de Janeiro,7400
Belo Horizonte,4462
Curitiba,3861
Brasília,3218
Porto Alegre,3208
Florianópolis,2643
Campinas,2546
Santos,1927


5. Categorias: 63,929 linhas (0 removidas)


## 6. Coorte anterior ao log

Antes de 07/2024 o arquivo quase só tem contas que viraram assinantes (98,7%, contra 46,1% depois): as contas grátis
daquela época não estão aqui. As linhas ficam, marcadas em `cadastro_antes_do_log`, para que cada análise decida se as usa.

In [8]:
base["cadastro_antes_do_log"] = base["data_cadastro"] < INICIO_LOG
display(base.groupby(base["data_cadastro"].dt.year.rename("ano de cadastro"))["cadastro_antes_do_log"].agg(["size", "sum"])
        .rename(columns={"size": "contas", "sum": "antes do log"}))
base = registrar("6. Coorte anterior ao log", "cadastro_antes_do_log = data_cadastro < 2024-07-01", base)

,contas,antes do log
ano de cadastro,,
2020,436,436
2021,1358,1358
2022,1314,1314
2023,1423,1423
2024,11831,669
2025,25770,0
2026,21797,0


6. Coorte anterior ao log: 63,929 linhas (0 removidas)


## 7. Conferências

- 63.929 contas, `user_id` único;
- o conjunto de contas é o mesmo do `onboarding.csv` (63.929 contas, uma por conta);
- todo `user_id` das tabelas já limpas e dos brutos de pagamentos, atividades e amizades existe aqui.

In [9]:
onboarding = pd.read_csv(BRUTOS / "onboarding.csv", dtype=str, keep_default_na=False, usecols=["user_id"])
pagamentos = pd.read_csv(BRUTOS / "pagamentos.csv", dtype=str, keep_default_na=False, usecols=["user_id"])
amizades = pd.read_csv(BRUTOS / "amizades.csv", dtype=str, keep_default_na=False, usecols=["seguidor_id", "seguido_id"])
atividades = pd.read_parquet(BRUTOS / "atividades.parquet", columns=["user_id"])
assinaturas_limpa = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet", columns=["user_id"])
eventos_limpo = pd.read_parquet(PROCESSADOS / "eventos_assinatura_limpo.parquet", columns=["user_id"])

ids = set(base["user_id"])
conferencias = {
    "contas na tabela limpa": (len(base), 63_929),
    "user_id repetido": (int(base["user_id"].duplicated().sum()), 0),
    "contas do onboarding fora da tabela limpa": (len(set(onboarding["user_id"]) - ids), 0),
    "contas da tabela limpa sem onboarding": (len(ids - set(onboarding["user_id"])), 0),
    "assinaturas_limpa: contas fora": (len(set(assinaturas_limpa["user_id"]) - ids), 0),
    "eventos_assinatura_limpo: contas fora": (len(set(eventos_limpo["user_id"]) - ids), 0),
    "pagamentos: contas fora": (len(set(pagamentos["user_id"]) - ids), 0),
    "atividades: contas fora": (len(set(atividades["user_id"]) - ids), 0),
    "amizades: contas fora": (len((set(amizades["seguidor_id"]) | set(amizades["seguido_id"])) - ids), 0),
    "cadastro depois do corte": (int((base["data_cadastro"] > CORTE).sum()), 0),
}
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["ok"].all()

,obtido,esperado,ok
contas na tabela limpa,63929,63929,True
user_id repetido,0,0,True
contas do onboarding fora da tabela limpa,0,0,True
contas da tabela limpa sem onboarding,0,0,True
assinaturas_limpa: contas fora,0,0,True
eventos_assinatura_limpo: contas fora,0,0,True
pagamentos: contas fora,0,0,True
atividades: contas fora,0,0,True
amizades: contas fora,0,0,True
cadastro depois do corte,0,0,True


## 8. Saídas e integridade dos brutos

In [10]:
COLUNAS = ["user_id", "data_cadastro", "data_nascimento", "idade_no_corte", "sexo", "cidade", "uf", "cep_prefixo",
           "canal_aquisicao", "sistema", "versao_app", "relogio_conectado", "cadastro_antes_do_log"]
limpa = base[COLUNAS].reset_index(drop=True)
for col in limpa.columns[limpa.dtypes == object]:
    limpa[col] = limpa[col].astype(object).where(limpa[col].notna(), None)   # ausente sempre como None
assert "email" not in limpa.columns

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / "usuarios_limpa.parquet"
limpa.to_parquet(destino, index=False)

relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_usuarios_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame().assign(ausentes=limpa.isna().sum()))
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\usuarios_limpa.parquet: 63,929 linhas x 13 colunas


,tipo,ausentes
user_id,object,0
data_cadastro,datetime64[ns],0
data_nascimento,datetime64[ns],3834
idade_no_corte,Int64,3834
sexo,object,3089
cidade,object,0
uf,object,0
cep_prefixo,object,1872
canal_aquisicao,object,0
sistema,object,0


,passo,regra,linhas,removidas
0,0. Arquivo bruto,usuarios.csv sem filtro,64948,0
1,1. Sem contas de teste,e-mail não termina em @vitaliza.com.br,64888,60
2,2. Sem contas-cópia,e-mail com maiúscula (a conta mais nova de um ...,63929,959
3,3. Data de cadastro,"só a data, em horário de Brasília",63929,0
4,4. Nascimento e idade,"1900-01-01, 2031-05-12 e vazio viram ausente; ...",63929,0
5,5. Categorias,"sexo, sistema, cidade e cep_prefixo padronizados",63929,0
6,6. Coorte anterior ao log,cadastro_antes_do_log = data_cadastro < 2024-0...,63929,0


dados/brutos/ inalterado: True
